# Tumor population dynamics on a tyssue 2D epithelium

_Investigation `tumor-tyssue` — coder reproduction notebook._

**Question.** Can a non-spatial SBML tumor-population model (BioModels BIOMD0000000903, run in COPASI) drive discrete cell birth/death/differentiation on a tyssue 2D epithelial sheet, and how does the resulting tissue differ from a baseline epithelium?

A demonstration that a non-spatial breast-cancer population ODE (run in COPASI) can drive real, discrete vertex-model mechanics — cell division and apoptotic extrusion — on a 2D tyssue epithelial sheet, and that the coupled tissue diverges cleanly from a mechanics-only control.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/vivarium-tyssue').is_dir():
    REPO = Path('/Users/eranagmon/code/vivarium-tyssue')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from vivarium_tyssue.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---
from scripts.run_study_sims import run_study
from scripts.render_study_viz import _render_one

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `tumor-composite`

**Question.** Can the COPASI BIOMD0000000903 tumor-population ODE drive real, discrete cell division and apoptotic extrusion on a tyssue 2D epithelial sheet — a single seeded focus growing out into one clonal patch?

**Objective.** Build the COPASI-coupled tumor composite and demonstrate, through five visualizations, that BIOMD0000000903 drives discrete cell-fate dynamics on a tyssue 2D epithelial sheet.

**Hypothesis.** Under flux-driven coupling, a small seeded tumor focus expands while healthy cells are progressively converted/killed, producing a clear shift in cell-type composition over the run.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `tumor` | `tumor` | 150 | interval=0.01 |

Declared parameter sets (`study.yaml` variants):

- **reference** — `interval=0.01`, `division_crit=2.0`, `apoptosis_crit=2.0`, `seed={'tumor': 1, 'stem': 0}`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `tumor`** — `spec_tumor` (a plain, editable dict)


In [ ]:
from pbg_superpowers.composite_spec import load_spec
spec_tumor = load_spec(REPO / 'vivarium_tyssue/composites/tumor.composite.yaml')
describe_spec(spec_tumor)

In [ ]:
# === Edit parameters for composite 'tumor' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_tumor['parameters']['interval']['default'] = 0.01

# process 'Tyssue'  (local:EulerSolver)
# spec_tumor['state']['Tyssue']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_tumor['state']['Tyssue']['config']['name'] = 'Tumor Epithelium 2D'
spec_tumor['state']['Tyssue']['config']['eptm'] = 'workspace/datasets/test_square.hf5'
spec_tumor['state']['Tyssue']['config']['tissue_type'] = 'Sheet'
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['area_elasticity'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['prefered_area'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['perimeter_elasticity'] = 0.1
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['prefered_perimeter'] = 3.6
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['cell_type'] = 'healthy'
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['is_alive'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['edge_df']['line_tension'] = 0.0
spec_tumor['state']['Tyssue']['config']['parameters']['edge_df']['is_active'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['vert_df']['viscosity'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['vert_df']['is_alive'] = 1.0
spec_tumor['state']['Tyssue']['config']['geom'] = 'SheetGeometry'
spec_tumor['state']['Tyssue']['config']['effectors'] = ['LineTension', 'FaceAreaElasticity', 'PerimeterElasticity']
spec_tumor['state']['Tyssue']['config']['ref_effector'] = 'FaceAreaElasticity'
spec_tumor['state']['Tyssue']['config']['factory'] = 'model_factory'
spec_tumor['state']['Tyssue']['config']['settings']['threshold_length'] = 0.03
spec_tumor['state']['Tyssue']['config']['auto_reconnect'] = True

# process 'TumorCoupling'  (local:TumorCoupling)
# spec_tumor['state']['TumorCoupling']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_tumor['state']['TumorCoupling']['config']['model_source'] = 'workspace/datasets/BIOMD0000000903.xml'
spec_tumor['state']['TumorCoupling']['config']['copasi_time'] = 1.0
spec_tumor['state']['TumorCoupling']['config']['copasi_intervals'] = 10
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['tumor'] = 'Induction of tumor cell'
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['healthy'] = 'Increase in the healthy cell in the system'
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['stem'] = 'Formation of Stem cell'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['tumor'] = 'Removal of tumor cell y immune cell and other immune response'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['healthy'] = 'Decrase of healthy cell due to cancer'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['stem'] = 'Removal of stem cell from the system'
spec_tumor['state']['TumorCoupling']['config']['scales']['tumor_births'] = 2e-06
spec_tumor['state']['TumorCoupling']['config']['scales']['tumor_deaths'] = 8e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['healthy_births'] = 6e-08
spec_tumor['state']['TumorCoupling']['config']['scales']['healthy_deaths'] = 1.2e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['stem_births'] = 6e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['stem_deaths'] = 2e-05
spec_tumor['state']['TumorCoupling']['config']['geom'] = 'SheetGeometry'
spec_tumor['state']['TumorCoupling']['config']['dt'] = 1.0
spec_tumor['state']['TumorCoupling']['config']['growth_rate'] = 0.1
spec_tumor['state']['TumorCoupling']['config']['shrink_rate'] = 0.1
spec_tumor['state']['TumorCoupling']['config']['division_crit'] = 2.0
spec_tumor['state']['TumorCoupling']['config']['apoptosis_crit'] = 2.0
spec_tumor['state']['TumorCoupling']['config']['seed']['tumor'] = 1
spec_tumor['state']['TumorCoupling']['config']['seed']['stem'] = 0
spec_tumor['state']['TumorCoupling']['config']['topology_ops'] = True

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tumor-composite ===
STUDY = 'tumor-composite'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_tumor = 150
INTERVAL_tumor = 0.01

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # sim 'tumor' <- edited composite spec spec_tumor ('tumor')
        run_study(STUDY, 'tumor', spec_tumor, STEPS_tumor, INTERVAL_tumor)
    print(f'ran 1 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Tumor sheet snapshots**


In [ ]:
# Tumor sheet snapshots
show_viz(_render_one('local:TissueSheetSnapshots', {'title': 'Tumor sheet snapshots', 'coords': ['x', 'y'], 'n_panels': 6, 'caption': "Six stills from the run, faces colored by cell type and drawn with tyssue's own sheet_view (the same renderer as the animation, so stills and movie agree). A SINGLE tumor cell seeded at the sheet centre (red) grows out into one contiguous clonal patch while the sheet itself grows — there are visibly more cells at the end than at t=0 because births are real cell divisions that add faces to the mesh, not recolorings. At interval 0.01 with division_crit 2.0 the cells stay compact and relaxed (no overlap) as the focus expands."}, RUNS_DB, STUDY_YAML))

**Tumor sheet animation**


In [ ]:
# Tumor sheet animation
show_viz(_render_one('local:TissueSheetGif', {'title': 'Tumor sheet animation', 'coords': ['x', 'y'], 'num_frames': 60, 'caption': 'The full run animated from the same tyssue mesh history. Watch the tissue gain cells (cell_division) and tumor patches spread; transient black cells are mid- extrusion (apoptosis_extrusion) before they are removed from the sheet.'}, RUNS_DB, STUDY_YAML))

**Cell births over time**


In [ ]:
# Cell births over time
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Cell births over time (real cell-division events)', 'observables': ['tumor_births', 'healthy_births', 'stem_births'], 'caption': 'Each point is the count of discrete birth events fired that step, driven by the COPASI model\'s per-reaction fluxes (e.g. "Induction of tumor cell") scaled through a fractional accumulator. Under topology_ops every birth is a real tyssue cell_division — the cell splits and the mesh gains a face. Over the 150-step run these sum to 175 divisions (173 tumor, 2 healthy, 0 stem — the single seed grows purely by clonal division); the trace is spiky because events are integer and fire only when a flux\'s accumulator crosses 1.0.'}, RUNS_DB, STUDY_YAML))

**Cell deaths over time**


In [ ]:
# Cell deaths over time
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Cell deaths over time (apoptosis_extrusion events)', 'observables': ['tumor_deaths', 'healthy_deaths', 'stem_deaths'], 'caption': 'Per-step death events under flux control. Each is a real apoptosis_extrusion: the cell shrinks to a critical area and is removed from the sheet via remove_face (67 extrusions over the run — 60 tumor, 7 healthy). Because extruded cells leave the mesh entirely, deaths show up here as events, not as a "dead" cell count.'}, RUNS_DB, STUDY_YAML))

**Cell types over time**


In [ ]:
# Cell types over time
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Cell composition and tissue size over time', 'observables': ['total_count', 'tumor_count', 'healthy_count', 'healthy_fraction'], 'caption': 'total_count rises 206→312 as the tumor proliferates — direct evidence of real division-driven growth. Healthy absolute count barely falls (206→200) because the epithelium itself keeps dividing, so the displacement signal is the healthy FRACTION, which drops 1.00→0.64. dead_count is omitted: it is identically 0 under topology_ops (apoptotic cells extrude rather than becoming a "dead" type).'}, RUNS_DB, STUDY_YAML))

**Tumor clone growth**


In [ ]:
# Tumor clone growth
show_viz(_render_one('local:TumorCloneGrowth', {'title': 'Tumor clone growth — size and outward spread', 'sources': ['tumor'], 'caption': 'Quantifies the snapshots: the tumor cell count (red) climbs steadily 0→112 while the clone\'s spread radius (orange dashed — the mean distance of tumor faces from their own centroid) rises early then plateaus near 2.0 even as the count keeps growing. One tight, smoothly-growing radius is the signature of a SINGLE focus expanding and filling in; a scatter of separate foci would show a large, jumpy radius from the first step. This is the quantitative read-out of "seed a single tumor and have it grow out."'}, RUNS_DB, STUDY_YAML))

**Cell area over time**


In [ ]:
# Cell area over time
show_viz(_render_one('local:CellAreaOverTime', {'title': 'Cell-area distribution over time (mesh stays relaxed — no overlap)', 'sources': ['tumor'], 'prefered_area': 1.0, 'division_crit': 2.0, 'caption': 'Evidence the sheet relaxes between divisions rather than overlapping. The healthy median face area holds at the prefered area (~1.0); the tumor median sits a little lower (~0.9, freshly-divided daughters); and crucially the 5th-percentile area (the smallest faces on the whole sheet, grey dotted) stays well above zero throughout. Cell overlap would show up as faces collapsing toward zero area — they do not. Reference lines mark the prefered area (1.0) and the division critical area (2.0) a cell must reach before it splits.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| tumor_expansion | kind=last path=tumor_count | op gt threshold 10 |
| tissue_growth_by_division | kind=last path=total_count | op gt threshold 206 |
| healthy_displacement | kind=last path=healthy_fraction | op lt threshold 0.8 |
| apoptotic_extrusion | kind=series path=tumor_deaths | op nonzero |
| flux_driven_events | kind=series path=tumor_births | op nonzero |


## Study: `tumor-vs-baseline`

**Question.** Does adding the COPASI tumor coupling make the epithelial sheet diverge from an otherwise-identical mechanics-only baseline — in tissue size, composition, and morphology?

**Objective.** Compare the COPASI-coupled tumor composite against a mechanics-only baseline epithelial sheet of identical geometry, isolating the effect of the tumor coupling.

**Hypothesis.** With tumor coupling off (baseline), the sheet stays uniformly healthy; with it on (tumor), cell-type composition shifts toward tumor and dead cells while healthy cells decline — a clear, controlled divergence.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `epithelium_2d` | 150 | interval=0.01 |
| `tumor` | `tumor` | 150 | interval=0.01 |

Declared parameter sets (`study.yaml` variants):

- **tumor-coupling** — `composite=vivarium_tyssue.composites.tumor`, `interval=0.01`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `epithelium_2d`** — `spec_epithelium_2d` (a plain, editable dict)


In [ ]:
from pbg_superpowers.composite_spec import load_spec
spec_epithelium_2d = load_spec(REPO / 'vivarium_tyssue/composites/epithelium_2d.composite.yaml')
describe_spec(spec_epithelium_2d)

In [ ]:
# === Edit parameters for composite 'epithelium_2d' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_epithelium_2d['parameters']['interval']['default'] = 0.01

# process 'Tyssue'  (local:EulerSolver)
# spec_epithelium_2d['state']['Tyssue']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_epithelium_2d['state']['Tyssue']['config']['name'] = 'Epithelium 2D'
spec_epithelium_2d['state']['Tyssue']['config']['eptm'] = 'workspace/datasets/test_square.hf5'
spec_epithelium_2d['state']['Tyssue']['config']['tissue_type'] = 'Sheet'
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['area_elasticity'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['prefered_area'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['perimeter_elasticity'] = 0.1
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['prefered_perimeter'] = 3.6
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['cell_type'] = 'healthy'
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['face_df']['is_alive'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['edge_df']['line_tension'] = 0.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['edge_df']['is_active'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['vert_df']['viscosity'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['parameters']['vert_df']['is_alive'] = 1.0
spec_epithelium_2d['state']['Tyssue']['config']['geom'] = 'SheetGeometry'
spec_epithelium_2d['state']['Tyssue']['config']['effectors'] = ['LineTension', 'FaceAreaElasticity', 'PerimeterElasticity']
spec_epithelium_2d['state']['Tyssue']['config']['ref_effector'] = 'FaceAreaElasticity'
spec_epithelium_2d['state']['Tyssue']['config']['factory'] = 'model_factory'
spec_epithelium_2d['state']['Tyssue']['config']['settings']['threshold_length'] = 0.03
spec_epithelium_2d['state']['Tyssue']['config']['auto_reconnect'] = True

**Composite `tumor`** — `spec_tumor` (a plain, editable dict)


In [ ]:
from pbg_superpowers.composite_spec import load_spec
spec_tumor = load_spec(REPO / 'vivarium_tyssue/composites/tumor.composite.yaml')
describe_spec(spec_tumor)

In [ ]:
# === Edit parameters for composite 'tumor' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_tumor['parameters']['interval']['default'] = 0.01

# process 'Tyssue'  (local:EulerSolver)
# spec_tumor['state']['Tyssue']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_tumor['state']['Tyssue']['config']['name'] = 'Tumor Epithelium 2D'
spec_tumor['state']['Tyssue']['config']['eptm'] = 'workspace/datasets/test_square.hf5'
spec_tumor['state']['Tyssue']['config']['tissue_type'] = 'Sheet'
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['area_elasticity'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['prefered_area'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['perimeter_elasticity'] = 0.1
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['prefered_perimeter'] = 3.6
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['cell_type'] = 'healthy'
spec_tumor['state']['Tyssue']['config']['parameters']['face_df']['is_alive'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['edge_df']['line_tension'] = 0.0
spec_tumor['state']['Tyssue']['config']['parameters']['edge_df']['is_active'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['vert_df']['viscosity'] = 1.0
spec_tumor['state']['Tyssue']['config']['parameters']['vert_df']['is_alive'] = 1.0
spec_tumor['state']['Tyssue']['config']['geom'] = 'SheetGeometry'
spec_tumor['state']['Tyssue']['config']['effectors'] = ['LineTension', 'FaceAreaElasticity', 'PerimeterElasticity']
spec_tumor['state']['Tyssue']['config']['ref_effector'] = 'FaceAreaElasticity'
spec_tumor['state']['Tyssue']['config']['factory'] = 'model_factory'
spec_tumor['state']['Tyssue']['config']['settings']['threshold_length'] = 0.03
spec_tumor['state']['Tyssue']['config']['auto_reconnect'] = True

# process 'TumorCoupling'  (local:TumorCoupling)
# spec_tumor['state']['TumorCoupling']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_tumor['state']['TumorCoupling']['config']['model_source'] = 'workspace/datasets/BIOMD0000000903.xml'
spec_tumor['state']['TumorCoupling']['config']['copasi_time'] = 1.0
spec_tumor['state']['TumorCoupling']['config']['copasi_intervals'] = 10
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['tumor'] = 'Induction of tumor cell'
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['healthy'] = 'Increase in the healthy cell in the system'
spec_tumor['state']['TumorCoupling']['config']['birth_fluxes']['stem'] = 'Formation of Stem cell'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['tumor'] = 'Removal of tumor cell y immune cell and other immune response'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['healthy'] = 'Decrase of healthy cell due to cancer'
spec_tumor['state']['TumorCoupling']['config']['death_fluxes']['stem'] = 'Removal of stem cell from the system'
spec_tumor['state']['TumorCoupling']['config']['scales']['tumor_births'] = 2e-06
spec_tumor['state']['TumorCoupling']['config']['scales']['tumor_deaths'] = 8e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['healthy_births'] = 6e-08
spec_tumor['state']['TumorCoupling']['config']['scales']['healthy_deaths'] = 1.2e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['stem_births'] = 6e-07
spec_tumor['state']['TumorCoupling']['config']['scales']['stem_deaths'] = 2e-05
spec_tumor['state']['TumorCoupling']['config']['geom'] = 'SheetGeometry'
spec_tumor['state']['TumorCoupling']['config']['dt'] = 1.0
spec_tumor['state']['TumorCoupling']['config']['growth_rate'] = 0.1
spec_tumor['state']['TumorCoupling']['config']['shrink_rate'] = 0.1
spec_tumor['state']['TumorCoupling']['config']['division_crit'] = 2.0
spec_tumor['state']['TumorCoupling']['config']['apoptosis_crit'] = 2.0
spec_tumor['state']['TumorCoupling']['config']['seed']['tumor'] = 1
spec_tumor['state']['TumorCoupling']['config']['seed']['stem'] = 0
spec_tumor['state']['TumorCoupling']['config']['topology_ops'] = True

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tumor-vs-baseline ===
STUDY = 'tumor-vs-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_baseline = 150
INTERVAL_baseline = 0.01
STEPS_tumor = 150
INTERVAL_tumor = 0.01

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # sim 'baseline' <- edited composite spec spec_epithelium_2d ('epithelium_2d')
        run_study(STUDY, 'baseline', spec_epithelium_2d, STEPS_baseline, INTERVAL_baseline)
        # sim 'tumor' <- edited composite spec spec_tumor ('tumor')
        run_study(STUDY, 'tumor', spec_tumor, STEPS_tumor, INTERVAL_tumor)
    print(f'ran 2 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Tissue size and composition — tumor vs baseline**


In [ ]:
# Tissue size and composition — tumor vs baseline
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Tissue size and composition — tumor vs baseline', 'observables': ['total_count', 'tumor_count', 'healthy_fraction'], 'sources': ['tumor', 'baseline'], 'caption': 'The controlled contrast. The baseline arm holds flat — total_count stays 206, tumor 0, healthy_fraction 1.00 (mechanics only, no events). The tumor arm grows total_count 206→312 by real cell division, tumor rises 0→112, and healthy_fraction falls to 0.64. The only difference between the arms is the COPASI coupling.'}, RUNS_DB, STUDY_YAML))

**Births and deaths — tumor vs baseline**


In [ ]:
# Births and deaths — tumor vs baseline
show_viz(_render_one('local:TimeSeriesFromObservables', {'title': 'Births & deaths — tumor vs baseline', 'observables': ['tumor_births', 'healthy_deaths'], 'sources': ['tumor', 'baseline'], 'caption': "Per-step cell-division (tumor_births) and apoptosis-extrusion (healthy_deaths) events. The baseline arm fires none; the tumor arm's events are flux-driven by the SBML model. Cumulatively the tumor arm reaches 175 divisions and 67 extrusions."}, RUNS_DB, STUDY_YAML))

**Baseline sheet snapshots**


In [ ]:
# Baseline sheet snapshots
show_viz(_render_one('local:TissueSheetSnapshots', {'title': 'Baseline sheet snapshots', 'coords': ['x', 'y'], 'n_panels': 6, 'sources': ['baseline'], 'caption': 'Mechanics-only control (tyssue sheet_view). The sheet only relaxes — no new cells, no extrusions, uniformly healthy (blue) throughout. Contrast with the tumor arm below.'}, RUNS_DB, STUDY_YAML))

**Tumor sheet snapshots**


In [ ]:
# Tumor sheet snapshots
show_viz(_render_one('local:TissueSheetSnapshots', {'title': 'Tumor sheet snapshots', 'coords': ['x', 'y'], 'n_panels': 6, 'sources': ['tumor'], 'caption': 'Tumor arm (tyssue sheet_view). A single tumor cell seeded at the centre grows out into one contiguous red clone as the COPASI fluxes drive real divisions, and the sheet gains cells — the divergence from the static baseline above.'}, RUNS_DB, STUDY_YAML))

**Tumor sheet animation**


In [ ]:
# Tumor sheet animation
show_viz(_render_one('local:TissueSheetGif', {'title': 'Tumor sheet animation', 'coords': ['x', 'y'], 'num_frames': 60, 'sources': ['tumor'], 'caption': 'The requested video for this comparison: the tumor arm animated from its tyssue mesh history, showing the sheet growing by cell division and a single central tumor clone spreading outward over the 150-step run (transient black cells are mid-extrusion).'}, RUNS_DB, STUDY_YAML))

**Baseline sheet animation**


In [ ]:
# Baseline sheet animation
show_viz(_render_one('local:TissueSheetGif', {'title': 'Baseline sheet animation', 'coords': ['x', 'y'], 'num_frames': 60, 'sources': ['baseline'], 'caption': 'The baseline arm animated for side-by-side comparison — the mesh only relaxes mechanically; no cells are added or removed.'}, RUNS_DB, STUDY_YAML))

**Tumor clone growth (tumor arm)**


In [ ]:
# Tumor clone growth (tumor arm)
show_viz(_render_one('local:TumorCloneGrowth', {'title': 'Tumor clone growth — size and outward spread (tumor arm)', 'sources': ['tumor'], 'caption': "The tumor arm's clone quantified: tumor cell count (red) rises 0→112 while the clone spread radius (orange dashed) grows then plateaus — one compact focus expanding outward. The baseline arm has no tumor cells, so this metric is flat-zero there and is shown for the tumor arm only; it is the quantitative counterpart to the side-by-side snapshots above."}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| baseline_stays_uniform | kind=last path=healthy_count | op eq threshold 206 |
| tumor_arm_diverges | kind=last path=tumor_count | op gt threshold 10 |
| tumor_arm_grows_tissue | kind=last path=total_count | op gt threshold 206 |
| healthy_fraction_gap | kind=last path=healthy_fraction | op lt threshold 0.8 |


## Open decisions
- Expert opinion on whether the flux→event scales should be calibrated to data (they are currently tuned for a watchable cadence) before any quantitative claim.
- Whether to pursue 3D / curved geometries now that real topology ops run on the fork.
